#### Imports

In [1]:
import pandas as pd
import numpy as np

### Read csv files and merging 

In [2]:
# timeline - one year (January to November 2024)
Jotstar_content_consumption = pd.read_csv('../datasets/Jotstar_content_consumption.csv')

Jotstar_contents = pd.read_csv('../datasets/Jotstar_contents.csv')

Jotstar_subscribers = pd.read_csv('../datasets/Jotstar_subscribers.csv')

Liocinema_content_consumption = pd.read_csv('../datasets/liocinema_content_consumption.csv')

Liocinema_contents = pd.read_csv('../datasets/liocinema_contents.csv')

Liocinema_subscribers = pd.read_csv('../datasets/liocinema_subscribers.csv')

In [3]:
# subscribers merging
Jotstar_subscribers['platform'] = 'Jotstar'
Liocinema_subscribers['platform'] = 'Liocinema'

subscribers = pd.merge(Jotstar_subscribers, Liocinema_subscribers, how='outer')
subscribers.head()

,user_id,age_group,city_tier,subscription_date,subscription_plan,last_active_date,plan_change_date,new_subscription_plan,platform
0,UIDJS0000751588f,18-24,Tier 1,2024-06-10,Premium,NaN,NaN,NaN,Jotstar
1,UIDJS000093eeb86,18-24,Tier 1,2024-11-09,Free,NaN,NaN,NaN,Jotstar
2,UIDJS00010d7fa1e,25-34,Tier 1,2024-08-08,Free,NaN,NaN,NaN,Jotstar
3,UIDJS00013411a85,35-44,Tier 2,2024-05-31,VIP,NaN,NaN,NaN,Jotstar
4,UIDJS0003a3f54cf,35-44,Tier 1,2024-09-20,Premium,NaN,NaN,NaN,Jotstar


In [4]:
# content consumption merging
Jotstar_content_consumption['platform'] = 'Jotstar'
Liocinema_content_consumption['platform'] = 'Liocinema'

Content_consumption = pd.merge(Jotstar_content_consumption, Liocinema_content_consumption, how='outer')
Content_consumption.head(), Content_consumption.tail()

(            user_id device_type  total_watch_time_mins platform
 0  UIDJS0000751588f      Laptop                   8019  Jotstar
 1  UIDJS0000751588f      Mobile                  20111  Jotstar
 2  UIDJS0000751588f          TV                  11997  Jotstar
 3  UIDJS000093eeb86      Laptop                    885  Jotstar
 4  UIDJS000093eeb86      Mobile                   2014  Jotstar,
                  user_id device_type  total_watch_time_mins   platform
 564607  UIDLCffffbb55ff5          TV                    121  Liocinema
 564608  UIDLCffffc6f6db0      Laptop                    934  Liocinema
 564609  UIDLCffffc6f6db0      Mobile                   5388  Liocinema
 564610  UIDLCffffc6f6db0          TV                    944  Liocinema
 564611  UIDLCffffe3b01f9      Mobile                    486  Liocinema)

In [5]:
# content dataset merging 
Jotstar_contents['platform'] = 'Jotstar'
Liocinema_contents['platform'] = 'Liocinema'

contents = pd.merge(Jotstar_contents, Liocinema_contents, how='outer')
contents.head(), contents.tail()

(       content_id content_type language   genre  run_time platform
 0  CJSMBEACT2e633        Movie  Bengali  Action        90  Jotstar
 1  CJSMBEACT34aec        Movie  Bengali  Action       135  Jotstar
 2  CJSMBEACT83b46        Movie  Bengali  Action       120  Jotstar
 3  CJSMBECOM12e7a        Movie  Bengali  Comedy       120  Jotstar
 4  CJSMBECOM5431a        Movie  Bengali  Comedy       135  Jotstar,
           content_id content_type language     genre  run_time   platform
 3605  CLCSTEROMe2fe2       Series   Telugu   Romance        45  Liocinema
 3606  CLCSTETHR03dcc       Series   Telugu  Thriller        30  Liocinema
 3607  CLCSTETHR99295       Series   Telugu  Thriller        45  Liocinema
 3608  CLCSTETHRa1536       Series   Telugu  Thriller        20  Liocinema
 3609  CLCSTETHRf659a       Series   Telugu  Thriller        45  Liocinema)

In [6]:
subscribers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228066 entries, 0 to 228065
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   user_id                228066 non-null  object
 1   age_group              228066 non-null  object
 2   city_tier              228066 non-null  object
 3   subscription_date      228066 non-null  object
 4   subscription_plan      228066 non-null  object
 5   last_active_date       88957 non-null   object
 6   plan_change_date       32104 non-null   object
 7   new_subscription_plan  32104 non-null   object
 8   platform               228066 non-null  object
dtypes: object(9)
memory usage: 15.7+ MB


In [7]:
subscribers.isna().sum()

user_id                       0
age_group                     0
city_tier                     0
subscription_date             0
subscription_plan             0
last_active_date         139109
plan_change_date         195962
new_subscription_plan    195962
platform                      0
dtype: int64

### Merging of subscribers + content_consumption data

In [8]:
# convert the date column from object to datetime datetype
subscribers['last_active_date'] = pd.to_datetime(subscribers['last_active_date'], errors='coerce')
subscribers['plan_change_date'] = pd.to_datetime(subscribers['plan_change_date'], errors='coerce')
subscribers['subscription_date'] = pd.to_datetime(subscribers['subscription_date'], errors='coerce')

In [9]:
# print max and min dates of all date related column
print(f'Subscribers last active date: min => {subscribers['last_active_date'].min().date()}  max => {subscribers['last_active_date'].max().date()}')
print(f'Subscribers plan change date: min => {subscribers['plan_change_date'].min().date()}  max => {subscribers['plan_change_date'].max().date()}')
print(f'Subscribers subscription date: min => {subscribers['subscription_date'].min().date()}  max => {subscribers['subscription_date'].max().date()}')


Subscribers last active date: min => 2024-02-01  max => 2024-12-29
Subscribers plan change date: min => 2024-02-01  max => 2024-12-31
Subscribers subscription date: min => 2024-01-01  max => 2024-11-30


In [10]:
# sum total watch time of content by users
res_Content_consumption = Content_consumption.groupby(['user_id','platform'])['total_watch_time_mins'].sum().reset_index()
res_Content_consumption = res_Content_consumption.rename(columns={'total_watch_time_mins':'total_watch_time'})
res_Content_consumption

,user_id,platform,total_watch_time
0,UIDJS0000751588f,Jotstar,40127
1,UIDJS000093eeb86,Jotstar,3868
2,UIDJS00010d7fa1e,Jotstar,9913
3,UIDJS00013411a85,Jotstar,21797
4,UIDJS0003a3f54cf,Jotstar,11531
...,...,...,...
228061,UIDLCffff41ace17,Liocinema,251
228062,UIDLCffff85ea59a,Liocinema,6808
228063,UIDLCffffbb55ff5,Liocinema,1043
228064,UIDLCffffc6f6db0,Liocinema,7266


In [11]:
master_dataset = pd.merge(subscribers, res_Content_consumption, how = 'left')
master_dataset

,user_id,age_group,city_tier,subscription_date,subscription_plan,last_active_date,plan_change_date,new_subscription_plan,platform,total_watch_time
0,UIDJS0000751588f,18-24,Tier 1,2024-06-10,Premium,NaT,NaT,NaN,Jotstar,40127
1,UIDJS000093eeb86,18-24,Tier 1,2024-11-09,Free,NaT,NaT,NaN,Jotstar,3868
2,UIDJS00010d7fa1e,25-34,Tier 1,2024-08-08,Free,NaT,NaT,NaN,Jotstar,9913
3,UIDJS00013411a85,35-44,Tier 2,2024-05-31,VIP,NaT,NaT,NaN,Jotstar,21797
4,UIDJS0003a3f54cf,35-44,Tier 1,2024-09-20,Premium,NaT,NaT,NaN,Jotstar,11531
...,...,...,...,...,...,...,...,...,...,...
228061,UIDLCffff41ace17,18-24,Tier 2,2024-10-16,Free,2024-11-07,NaT,NaN,Liocinema,251
228062,UIDLCffff85ea59a,25-34,Tier 2,2024-08-09,Basic,NaT,NaT,NaN,Liocinema,6808
228063,UIDLCffffbb55ff5,18-24,Tier 2,2024-11-22,Free,NaT,NaT,NaN,Liocinema,1043
228064,UIDLCffffc6f6db0,18-24,Tier 1,2024-05-01,Basic,2024-10-10,NaT,NaN,Liocinema,7266


In [12]:
master_dataset['platform'].unique()
master_dataset['subscription_plan'].unique()

array(['Premium', 'Free', 'VIP', 'Basic'], dtype=object)

#### Feature extraction - creating new columns using subscriptions date history of users.

**Number of days the user stayed since last active date** 

It tells how much time users has spend on platform. It can be that users has spend whole year till the dataset end date (2024-12-31) or has left midway.
The last date mention tell users last activity on the platform. If the value is NaT / NaN then users is still active.
```
If last_active_date is EMPTY (still active user):
    days = 2024-12-31 - subscription_date
    → how long they've been active till end of dataset

If last_active_date EXISTS (churned user):
    days = last_active_date - subscription_date
    → how long they stayed before leaving
```

In [13]:
last_date = pd.Timestamp('2024-12-31') 

In [14]:
master_dataset['days_since_last_active'] = np.where(
    master_dataset['last_active_date'].isna(),
    (last_date - master_dataset['subscription_date']).dt.days, # still active
    (master_dataset['last_active_date'] - master_dataset['subscription_date'] ).dt.days # inactive
)
master_dataset 

,user_id,age_group,city_tier,subscription_date,subscription_plan,last_active_date,plan_change_date,new_subscription_plan,platform,total_watch_time,days_since_last_active
0,UIDJS0000751588f,18-24,Tier 1,2024-06-10,Premium,NaT,NaT,NaN,Jotstar,40127,204.0
1,UIDJS000093eeb86,18-24,Tier 1,2024-11-09,Free,NaT,NaT,NaN,Jotstar,3868,52.0
2,UIDJS00010d7fa1e,25-34,Tier 1,2024-08-08,Free,NaT,NaT,NaN,Jotstar,9913,145.0
3,UIDJS00013411a85,35-44,Tier 2,2024-05-31,VIP,NaT,NaT,NaN,Jotstar,21797,214.0
4,UIDJS0003a3f54cf,35-44,Tier 1,2024-09-20,Premium,NaT,NaT,NaN,Jotstar,11531,102.0
...,...,...,...,...,...,...,...,...,...,...,...
228061,UIDLCffff41ace17,18-24,Tier 2,2024-10-16,Free,2024-11-07,NaT,NaN,Liocinema,251,22.0
228062,UIDLCffff85ea59a,25-34,Tier 2,2024-08-09,Basic,NaT,NaT,NaN,Liocinema,6808,144.0
228063,UIDLCffffbb55ff5,18-24,Tier 2,2024-11-22,Free,NaT,NaT,NaN,Liocinema,1043,39.0
228064,UIDLCffffc6f6db0,18-24,Tier 1,2024-05-01,Basic,2024-10-10,NaT,NaN,Liocinema,7266,162.0


**How many days in new subscription plan done**

```
If plan_change_date is EMPTY (never changed plan):
    days = 2024-12-31 - subscription_date
    → how long they've been on original plan till end of dataset

If plan_change_date EXISTS (changed plan):
    days = plan_change_date - subscription_date
    → how many days before they switched their plan
```

**days_after_new_plan**

last_active_date and plan_change_date are not mention at a same time. Data of this columns are alternative.

In [15]:
master_dataset['days_in_new_plan'] = np.where(
    master_dataset['plan_change_date'].isna(), 0,
    (last_date - master_dataset['plan_change_date']).dt.days
)
master_dataset 

,user_id,age_group,city_tier,subscription_date,subscription_plan,last_active_date,plan_change_date,new_subscription_plan,platform,total_watch_time,days_since_last_active,days_in_new_plan
0,UIDJS0000751588f,18-24,Tier 1,2024-06-10,Premium,NaT,NaT,NaN,Jotstar,40127,204.0,0.0
1,UIDJS000093eeb86,18-24,Tier 1,2024-11-09,Free,NaT,NaT,NaN,Jotstar,3868,52.0,0.0
2,UIDJS00010d7fa1e,25-34,Tier 1,2024-08-08,Free,NaT,NaT,NaN,Jotstar,9913,145.0,0.0
3,UIDJS00013411a85,35-44,Tier 2,2024-05-31,VIP,NaT,NaT,NaN,Jotstar,21797,214.0,0.0
4,UIDJS0003a3f54cf,35-44,Tier 1,2024-09-20,Premium,NaT,NaT,NaN,Jotstar,11531,102.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
228061,UIDLCffff41ace17,18-24,Tier 2,2024-10-16,Free,2024-11-07,NaT,NaN,Liocinema,251,22.0,0.0
228062,UIDLCffff85ea59a,25-34,Tier 2,2024-08-09,Basic,NaT,NaT,NaN,Liocinema,6808,144.0,0.0
228063,UIDLCffffbb55ff5,18-24,Tier 2,2024-11-22,Free,NaT,NaT,NaN,Liocinema,1043,39.0,0.0
228064,UIDLCffffc6f6db0,18-24,Tier 1,2024-05-01,Basic,2024-10-10,NaT,NaN,Liocinema,7266,162.0,0.0


**days_before_new_plan**

In [16]:
master_dataset['days_before_new_plan'] = np.where(
    master_dataset['plan_change_date'].isna(),
    0,
    (master_dataset['plan_change_date'] - master_dataset['subscription_date']).dt.days
)
master_dataset 

,user_id,age_group,city_tier,subscription_date,subscription_plan,last_active_date,plan_change_date,new_subscription_plan,platform,total_watch_time,days_since_last_active,days_in_new_plan,days_before_new_plan
0,UIDJS0000751588f,18-24,Tier 1,2024-06-10,Premium,NaT,NaT,NaN,Jotstar,40127,204.0,0.0,0.0
1,UIDJS000093eeb86,18-24,Tier 1,2024-11-09,Free,NaT,NaT,NaN,Jotstar,3868,52.0,0.0,0.0
2,UIDJS00010d7fa1e,25-34,Tier 1,2024-08-08,Free,NaT,NaT,NaN,Jotstar,9913,145.0,0.0,0.0
3,UIDJS00013411a85,35-44,Tier 2,2024-05-31,VIP,NaT,NaT,NaN,Jotstar,21797,214.0,0.0,0.0
4,UIDJS0003a3f54cf,35-44,Tier 1,2024-09-20,Premium,NaT,NaT,NaN,Jotstar,11531,102.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
228061,UIDLCffff41ace17,18-24,Tier 2,2024-10-16,Free,2024-11-07,NaT,NaN,Liocinema,251,22.0,0.0,0.0
228062,UIDLCffff85ea59a,25-34,Tier 2,2024-08-09,Basic,NaT,NaT,NaN,Liocinema,6808,144.0,0.0,0.0
228063,UIDLCffffbb55ff5,18-24,Tier 2,2024-11-22,Free,NaT,NaT,NaN,Liocinema,1043,39.0,0.0,0.0
228064,UIDLCffffc6f6db0,18-24,Tier 1,2024-05-01,Basic,2024-10-10,NaT,NaN,Liocinema,7266,162.0,0.0,0.0


**There are no records of a users who has it's last active date after they have changed their plan in the whole dataset.**

This tells us that both columns are mutally exclusive (independent of each other). 

There are only users who has last_active with their original plan. OR The users who have changed their plan and are still active.

So those users who have last_active_date with their original plan are churned while those still on their original plan and active but not change in the plan are retained. 

In [17]:
master_dataset[master_dataset['last_active_date'].isna() & master_dataset['new_subscription_plan'].notna()] 

,user_id,age_group,city_tier,subscription_date,subscription_plan,last_active_date,plan_change_date,new_subscription_plan,platform,total_watch_time,days_since_last_active,days_in_new_plan,days_before_new_plan
12,UIDJS000cd6a7fe8,25-34,Tier 1,2024-09-18,Premium,NaT,2024-12-18,VIP,Jotstar,14815,104.0,13.0,91.0
19,UIDJS0019b04b32f,18-24,Tier 1,2024-02-11,VIP,NaT,2024-08-11,Free,Jotstar,44053,324.0,142.0,182.0
23,UIDJS001defd7e03,25-34,Tier 1,2024-01-18,VIP,NaT,2024-08-18,Free,Jotstar,36617,348.0,135.0,213.0
47,UIDJS003cb5a2cd7,35-44,Tier 1,2024-06-05,VIP,NaT,2024-10-05,Free,Jotstar,19926,209.0,87.0,122.0
58,UIDJS004c0eb4154,25-34,Tier 2,2024-02-26,VIP,NaT,2024-09-26,Free,Jotstar,35361,309.0,96.0,213.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
228016,UIDLCffeded3d2d5,18-24,Tier 3,2024-08-20,Basic,NaT,2024-11-20,Premium,Liocinema,6556,133.0,41.0,92.0
228019,UIDLCffef39a6cf8,35-44,Tier 1,2024-06-05,Free,NaT,2024-11-05,Basic,Liocinema,7485,209.0,56.0,153.0
228035,UIDLCfff3e7a22ad,35-44,Tier 1,2024-06-08,Free,NaT,2024-12-08,Premium,Liocinema,9664,206.0,23.0,183.0
228045,UIDLCfff830078b9,25-34,Tier 3,2024-08-22,Premium,NaT,2024-10-22,Free,Liocinema,5845,131.0,70.0,61.0


**upgrades and downgrade, retained category**
```
Churned — last_active_date is filled in AND plan_change_date is null. The user went inactive without switching plans.
Retained — both last_active_date and plan_change_date are null. User is still active on their original plan.
Downgraded — plan_change_date is filled AND new_subscription_plan rank < subscription_plan rank.
Upgraded — plan_change_date is filled AND new_subscription_plan rank > subscription_plan rank.

Category is used has upgraded, downgraded and same
Churned  = 0 # since user 
retained = 1
downgraded = 2
upgraded = 3
```
**last_active_date and plan_change_date are not mention at a same time. BOTH CANNOT GO HAND IN HAND. Data of this columns are alternative.**


In [18]:
subscription_plan_map = {'Free':1,'Basic':2,'Premium':3,'VIP':4, np.nan : 0}

master_dataset['subscription_plan_status'] = np.where(  
    master_dataset['new_subscription_plan'].isna(), 
    np.where(master_dataset['last_active_date'].isna(),'Retained','Churned'),
    np.where(
        master_dataset['new_subscription_plan'].map(subscription_plan_map) > master_dataset['subscription_plan'].map(subscription_plan_map),
        'Upgrade',
        'Downgrade'
    )
)

### Save the final merged dataset

In [19]:
master_dataset.to_csv('master_df.csv', index=False)

In [20]:
master_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228066 entries, 0 to 228065
Data columns (total 14 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   user_id                   228066 non-null  object        
 1   age_group                 228066 non-null  object        
 2   city_tier                 228066 non-null  object        
 3   subscription_date         228066 non-null  datetime64[ns]
 4   subscription_plan         228066 non-null  object        
 5   last_active_date          88957 non-null   datetime64[ns]
 6   plan_change_date          32104 non-null   datetime64[ns]
 7   new_subscription_plan     32104 non-null   object        
 8   platform                  228066 non-null  object        
 9   total_watch_time          228066 non-null  int64         
 10  days_since_last_active    228066 non-null  float64       
 11  days_in_new_plan          228066 non-null  float64       
 12  da

In [21]:
df_without_datetime = master_dataset.select_dtypes(exclude=['datetime64[ns]'])
df_without_datetime.to_csv('ott_merge.csv',index=False)

In [22]:
# read ott master data
ott = pd.read_csv('ott_merge.csv') 
ott

,user_id,age_group,city_tier,subscription_plan,new_subscription_plan,platform,total_watch_time,days_since_last_active,days_in_new_plan,days_before_new_plan,subscription_plan_status
0,UIDJS0000751588f,18-24,Tier 1,Premium,NaN,Jotstar,40127,204.0,0.0,0.0,Retained
1,UIDJS000093eeb86,18-24,Tier 1,Free,NaN,Jotstar,3868,52.0,0.0,0.0,Retained
2,UIDJS00010d7fa1e,25-34,Tier 1,Free,NaN,Jotstar,9913,145.0,0.0,0.0,Retained
3,UIDJS00013411a85,35-44,Tier 2,VIP,NaN,Jotstar,21797,214.0,0.0,0.0,Retained
4,UIDJS0003a3f54cf,35-44,Tier 1,Premium,NaN,Jotstar,11531,102.0,0.0,0.0,Retained
...,...,...,...,...,...,...,...,...,...,...,...
228061,UIDLCffff41ace17,18-24,Tier 2,Free,NaN,Liocinema,251,22.0,0.0,0.0,Churned
228062,UIDLCffff85ea59a,25-34,Tier 2,Basic,NaN,Liocinema,6808,144.0,0.0,0.0,Retained
228063,UIDLCffffbb55ff5,18-24,Tier 2,Free,NaN,Liocinema,1043,39.0,0.0,0.0,Retained
228064,UIDLCffffc6f6db0,18-24,Tier 1,Basic,NaN,Liocinema,7266,162.0,0.0,0.0,Churned
